# Visualising Turin — Dataset Analysis

This notebook analyses and visualises the four datasets collected for this project:
- **British Pathé** — 206 archival newsreel clips of Turin (1914–2000)
- **IMDb** — 286 films set in Turin (1904–2026)
- **FCTP Industrial Buildings** — 68 industrial buildings in Turin
- **GPT Dialogues** — 24 synthetic location dialogues (Franco × Nietzsche)

The visualisations are organised in three groups:
1. Within each dataset — understanding each source individually
2. Cross-dataset — comparing how Turin appears across different media
3. Semantic analysis — what themes and vocabularies run through the city's representations

All outputs are saved to subfolders of `OUTPUTS/`.

## 1. Setup

I define a consistent visual style across all charts — a minimal palette, clean typography, and no unnecessary decoration. All charts use the same colour family so they read as a coherent set when placed together in the report.

In [1]:
!pip install matplotlib seaborn pandas openpyxl adjustText -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\elena\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [29]:
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from collections import Counter
from pathlib import Path

# --- Paths ---
BASE_OUT   = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS"
PATHE_CSV  = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\BRITISH PATHE\VIDEOS.xlsx"
IMDB_CSV   = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\IMDB\films.csv"
FCTP_CSV   = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\INDUSTRIAL BUILDINGS\fctp_industrial_turin.csv"
DIAL_DIR   = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\APIs\dialogues"

OUT_PATHE  = os.path.join(BASE_OUT, "british_pathe")
OUT_IMDB   = os.path.join(BASE_OUT, "imdb")
OUT_FCTP   = os.path.join(BASE_OUT, "fctp")
OUT_DIAL   = os.path.join(BASE_OUT, "dialogues")
OUT_CROSS  = os.path.join(BASE_OUT, "cross_dataset")

for d in [OUT_PATHE, OUT_IMDB, OUT_FCTP, OUT_DIAL, OUT_CROSS]:
    os.makedirs(d, exist_ok=True)

# --- Global style ---
plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.size':         11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.25,
    'grid.linestyle':    '--',
    'figure.dpi':        150,
})

# --- Forest Hues palette ---
C_PATHE  = '#C0462B'   # burnt rust    — archive
C_IMDB   = '#1D6A72'   # deep teal     — cinema
C_FCTP   = '#2A9199'   # warm teal     — architecture
C_DIAL   = '#8B2E16'   # dark rust     — dialogue

PAL_WARM = ['#C0462B','#D06B52','#E0977A','#EFC3A8','#F8E8D8']
PAL_COOL = ['#0F3D42','#1D6A72','#2A9199','#60B8BE','#A8DADE']
PAL_SAGE = ['#0F3D42','#1D6A72','#2A9199','#60B8BE','#A8DADE']
PAL_MIX  = [C_PATHE, C_IMDB, C_FCTP, C_DIAL]

def save(fig, folder, name):
    path = os.path.join(folder, name)
    fig.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'Saved: {path}')

print('Setup complete.')

Setup complete.


## 2. Loading the datasets

In [30]:
# British Pathé
df_pathe = pd.read_excel(r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\BRITISH PATHE\VIDEOS.xlsx")
df_pathe.columns = ['title', 'video_page_url', 'mp4_url', 'thumbnail', 'short_summary']
df_pathe = df_pathe.iloc[1:].reset_index(drop=True)  # drop the row that had column names as data

print('Columns:', list(df_pathe.columns))
print('Rows:', len(df_pathe))
print(df_pathe['title'].head(3).tolist())

def extract_year(text):
    m = re.findall(r'\b(18\d{2}|19\d{2}|20\d{2})\b', str(text))
    return int(m[0]) if m else None

df_pathe['year']   = df_pathe['title'].apply(extract_year)
df_pathe['decade'] = (df_pathe['year'] // 10 * 10).where(df_pathe['year'].notna())

TOPICS = {
    'Labour / Strikes':   ['strike','worker','fiat','lay-off','union','workforce'],
    'Sport':              ['soccer','football','motor race','rally','sport','juventus','athletics'],
    'War / Military':     ['war','bomb','military','defence','prisoner','riot','troops'],
    'Royalty / Politics': ['royal','crown','prince','king','president','inaugur','politic'],
    'Industry / Cars':    ['car sales','fiat','industri','hydro','factory','produc','automobile'],
    'Culture / Events':   ['carnival','ball','festival','exhib','expo','motor show','fair'],
    'Crime / Unrest':     ['crime','kidnap','brigades','guerrilla','arrest','murder','prison'],
}

def get_topic(title):
    t = str(title).lower()
    for topic, kws in TOPICS.items():
        if any(k in t for k in kws):
            return topic
    return 'Other'

df_pathe['topic'] = df_pathe['title'].apply(get_topic)
print(f'British Pathé: {len(df_pathe)} clips | {df_pathe["year"].notna().sum()} with years')

# IMDb
df_imdb = pd.read_csv(IMDB_CSV)
df_imdb['year']   = pd.to_numeric(df_imdb['year'], errors='coerce')
df_imdb['decade'] = (df_imdb['year'] // 10 * 10).where(df_imdb['year'].notna()).astype('Int64')
print(f'IMDb: {len(df_imdb)} films | {df_imdb["genres"].notna().sum()} with genres')

# FCTP
df_fctp = pd.read_csv(FCTP_CSV)
print(f'FCTP: {len(df_fctp)} buildings')

# Dialogues
dial_files = sorted(glob.glob(os.path.join(DIAL_DIR, '*.txt')))
dialogues  = []
for p in dial_files:
    name = Path(p).stem
    with open(p, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    franco    = ' '.join(re.findall(r'Franco:(.*?)(?=Nietzsche:|$)', text, re.S))
    nietzsche = ' '.join(re.findall(r'Nietzsche:(.*?)(?=Franco:|$)', text, re.S))
    dialogues.append({'location': name, 'franco': franco.strip(),
                      'nietzsche': nietzsche.strip(), 'full': text})
df_dial = pd.DataFrame(dialogues)
print(f'Dialogues: {len(df_dial)} locations')

Columns: ['title', 'video_page_url', 'mp4_url', 'thumbnail', 'short_summary']
Rows: 205
['UK: SOCCER--LEEDS DRAW WITH JUVENTUS OF TURIN TO WIN FAIRE CUP BY NARROWEST POSSIBLE MARGIN. (1971)', 'TURIN - SOCCER INTERNATIONAL (1948)', 'TURIN / CATHERINETTES BALL (1939)']
British Pathé: 205 clips | 200 with years
IMDb: 286 films | 284 with genres
FCTP: 68 buildings
Dialogues: 23 locations


---
## 3. British Pathé Analysis

### 3.1 Clips per decade

This chart shows the rhythm of archival attention — when the camera was pointing at Turin. The concentration of clips reveals which decades generated the most internationally newsworthy events in the city.

In [31]:
decade_counts = df_pathe.dropna(subset=['decade']).groupby('decade').size()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(decade_counts.index.astype(int), decade_counts.values,
              color=C_PATHE, alpha=0.85, width=7, edgecolor='white', linewidth=0.5)

ax.set_xlabel('Decade', labelpad=8)
ax.set_ylabel('Number of clips', labelpad=8)
ax.set_title('British Pathé — Archival Clips of Turin by Decade', pad=14, fontweight='bold')
ax.set_xticks(decade_counts.index.astype(int))
ax.set_xticklabels([f"{int(d)}s" for d in decade_counts.index], rotation=45, ha='right')

for bar in bars:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, str(int(h)),
                ha='center', va='bottom', fontsize=9, color='#555')

fig.tight_layout()
save(fig, OUT_PATHE, 'pathe_clips_per_decade.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\british_pathe\pathe_clips_per_decade.png


### 3.2 Topic × Decade heatmap

This heatmap shows when each type of event was being filmed in Turin. It reads like a compressed political history — when the city was industrial, when it was at war, when it was in social conflict. The colour intensity shows the concentration of clips per topic per decade.

In [32]:
df_p = df_pathe.dropna(subset=['decade']).copy()
df_p['decade'] = df_p['decade'].astype(int)

pivot = df_p.groupby(['topic','decade']).size().unstack(fill_value=0)
# Normalise each row so topics of different sizes are comparable
pivot_norm = pivot.div(pivot.sum(axis=1), axis=0)

# Order topics by total count
topic_order = df_p['topic'].value_counts().index.tolist()
pivot_norm  = pivot_norm.reindex(topic_order)

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(pivot_norm, ax=ax,
            cmap='YlOrBr', linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'Proportion of clips', 'shrink': 0.6},
            annot=pivot.reindex(topic_order), fmt='d', annot_kws={'size': 8})

ax.set_xlabel('Decade', labelpad=8)
ax.set_ylabel('')
ax.set_title('British Pathé — Topic Distribution Across Decades\n(colour = proportion, numbers = raw count)',
             pad=14, fontweight='bold')
ax.set_xticklabels([f"{c.get_text()}s" for c in ax.get_xticklabels()], rotation=45, ha='right')

fig.tight_layout()
save(fig, OUT_PATHE, 'pathe_topic_decade_heatmap.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\british_pathe\pathe_topic_decade_heatmap.png


---
## 4. IMDb Analysis

### 4.1 Films per decade

Mirrors the Pathé chart and makes them directly comparable later. Shows when Turin became a significant location for cinema — and whether fictional representation followed or preceded archival documentation.

In [33]:
decade_counts_imdb = (df_imdb.dropna(subset=['decade'])
                      .groupby('decade').size())

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(decade_counts_imdb.index.astype(int),
              decade_counts_imdb.values,
              color=C_IMDB, alpha=0.85, width=7,
              edgecolor='white', linewidth=0.5)

ax.set_xlabel('Decade', labelpad=8)
ax.set_ylabel('Number of films', labelpad=8)
ax.set_title('IMDb — Films Set in Turin by Decade', pad=14, fontweight='bold')
ax.set_xticks(decade_counts_imdb.index.astype(int))
ax.set_xticklabels([f"{int(d)}s" for d in decade_counts_imdb.index],
                    rotation=45, ha='right')

for bar in bars:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, str(int(h)),
                ha='center', va='bottom', fontsize=9, color='#555')

fig.tight_layout()
save(fig, OUT_IMDB, 'imdb_films_per_decade.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\imdb\imdb_films_per_decade.png


### 4.2 Genre × Decade heatmap

Shows how the cinematic representation of Turin shifted over time. The 1970s crime thriller wave, the prevalence of documentary shorts in earlier decades, and the contemporary distribution of genres all become visible. This directly parallels the Pathé topic heatmap.

In [34]:
# Explode genres so each film-genre pair is one row
df_g = df_imdb.dropna(subset=['genres','decade']).copy()
df_g['decade'] = df_g['decade'].astype(int)
df_g['genre_list'] = df_g['genres'].str.split(', ')
df_exploded = df_g.explode('genre_list')
df_exploded['genre_list'] = df_exploded['genre_list'].str.strip()

# Keep top 12 genres
top_genres = df_exploded['genre_list'].value_counts().head(12).index.tolist()
df_top = df_exploded[df_exploded['genre_list'].isin(top_genres)]

pivot_g = df_top.groupby(['genre_list','decade']).size().unstack(fill_value=0)
pivot_g_norm = pivot_g.div(pivot_g.sum(axis=1), axis=0)

# Order by total count
genre_order = df_top['genre_list'].value_counts().index.tolist()
pivot_g_norm = pivot_g_norm.reindex(genre_order)
pivot_g      = pivot_g.reindex(genre_order)

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(pivot_g_norm, ax=ax,
            cmap='Blues', linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'Proportion of genre occurrences', 'shrink': 0.6},
            annot=pivot_g, fmt='d', annot_kws={'size': 8})

ax.set_xlabel('Decade', labelpad=8)
ax.set_ylabel('')
ax.set_title('IMDb — Genre Distribution Across Decades\n(colour = proportion, numbers = raw count)',
             pad=14, fontweight='bold')
ax.set_xticklabels([f"{c.get_text()}s" for c in ax.get_xticklabels()],
                    rotation=45, ha='right')

fig.tight_layout()
save(fig, OUT_IMDB, 'imdb_genre_decade_heatmap.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\imdb\imdb_genre_decade_heatmap.png


---
## 5. FCTP Industrial Buildings Analysis

### 5.1 Architectural period × style heatmap

The material skeleton of Turin. This chart shows which architectural styles emerged in which historical periods — revealing whether certain design approaches were tied to specific eras of industrial development.

In [35]:
def simplify_period(p):
    """Extract the primary period from compound period strings."""
    p = str(p)
    if 'Middle ages' in p:           return 'Middle Ages'
    if '17th' in p:                  return '17th Century'
    if '18th' in p:                  return '18th Century'
    if '19th' in p and '20th' not in p: return '19th Century'
    if "'10s and '20s" in p:         return "1910s–1920s"
    if "'30s and '40s" in p:         return "1930s–1940s"
    if "'50s" in p:                  return "1950s"
    if "'60s" in p:                  return "1960s"
    if "'70s" in p:                  return "1970s"
    if "'80s" in p:                  return "1980s"
    if "'90s" in p:                  return "1990s"
    if '21th' in p or '21st' in p:   return '21st Century'
    if '20th Century' in p:          return '20th Century'
    return 'Unknown'

def simplify_style(s):
    """Take the first style from compound style strings."""
    s = str(s).strip()
    if s == '-' or s == 'nan': return 'Unknown'
    return s.split(',')[0].strip()

df_f = df_fctp.copy()
df_f['period_simple'] = df_f['period'].apply(simplify_period)
df_f['style_simple']  = df_f['style'].apply(simplify_style)

period_order = ['Middle Ages','17th Century','18th Century','19th Century',
                '1910s–1920s','1930s–1940s','1950s','1960s','1970s',
                '1980s','1990s','20th Century','21st Century']

pivot_f = df_f.groupby(['style_simple','period_simple']).size().unstack(fill_value=0)
# Keep only period columns that exist
existing = [p for p in period_order if p in pivot_f.columns]
pivot_f  = pivot_f[existing]

# Sort styles by total count
pivot_f = pivot_f.loc[pivot_f.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot_f, ax=ax,
            cmap='Greens', linewidths=0.4, linecolor='white',
            annot=True, fmt='d', annot_kws={'size': 9},
            cbar_kws={'label': 'Number of buildings', 'shrink': 0.6})

ax.set_xlabel('Architectural Period', labelpad=8)
ax.set_ylabel('')
ax.set_title('FCTP Industrial Buildings — Style × Period',
             pad=14, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right')

fig.tight_layout()
save(fig, OUT_FCTP, 'fctp_style_period_heatmap.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\fctp\fctp_style_period_heatmap.png


---
## 6. Dialogue Analysis — Franco × Nietzsche

### 6.1 Character voice comparison

Franco and Nietzsche walk the same 24 locations but perceive them completely differently. This chart compares the most frequently used meaningful words by each character across all locations. Franco's vocabulary is sensory and material; Nietzsche's is philosophical and spatial. The contrast is the project's central argument made visible.

In [36]:
STOPWORDS = set((
    'the a an and or but if while of to in on at from for by with without '
    'this that these those is are was were be been being it its itself '
    'you your yours he she they we i me my mine him his her hers them '
    'their ours our as not no yes there here then than thus so just also '
    'very more most much many any some none one two three four five '
    'could would should might may can will do does did done have has had '
    'get got make made what where when who how which each every '
    'into out up down about across around before after through '
    'us let yet even still only though along oh well all its '
    'franco nietzsche turin torino city street place space '
    'walk feel sense know think see hear say tell want need '
    'something nothing everything perhaps though already seems seem '
    'like become becomes rather quite almost such here there '
    'must shall dear good great long way back now new old '
).split())

def top_words(text_series, n=20):
    all_text = ' '.join(text_series.dropna().tolist()).lower()
    words = re.findall(r'\b[a-z]{3,}\b', all_text)
    filtered = [w for w in words if w not in STOPWORDS]
    return Counter(filtered).most_common(n)

franco_top    = top_words(df_dial['franco'], 15)
nietzsche_top = top_words(df_dial['nietzsche'], 15)

f_words, f_vals = zip(*franco_top)
n_words, n_vals = zip(*nietzsche_top)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Franco
ax1.barh(range(len(f_words)), f_vals[::-1], color=C_DIAL, alpha=0.8)
ax1.set_yticks(range(len(f_words)))
ax1.set_yticklabels(f_words[::-1])
ax1.set_title('Franco Arno\nMost frequent words', fontweight='bold', pad=10)
ax1.set_xlabel('Word count')
ax1.invert_xaxis()
ax1.yaxis.set_label_position('right')
ax1.yaxis.tick_right()

# Nietzsche
ax2.barh(range(len(n_words)), n_vals[::-1], color=C_PATHE, alpha=0.8)
ax2.set_yticks(range(len(n_words)))
ax2.set_yticklabels(n_words[::-1])
ax2.set_title('Friedrich Nietzsche\nMost frequent words', fontweight='bold', pad=10)
ax2.set_xlabel('Word count')

fig.suptitle('Two Ways of Seeing Turin — Vocabulary Comparison',
             fontsize=13, fontweight='bold', y=1.01)
fig.tight_layout()
save(fig, OUT_DIAL, 'dialogue_character_voice_comparison.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\dialogues\dialogue_character_voice_comparison.png


---
## 7. Cross-Dataset Analysis

### 7.1 Shared timeline — Turin documented, fictionalised, and built

This is the central comparative visualisation. All three time-based datasets are placed on the same axis — archival clips, films, and building construction periods. The chart shows how Turin was built, then documented, then fictionalised — and how these waves of representation overlap and diverge.

In [37]:
# Prepare decade counts for each dataset
pathe_d  = df_pathe.dropna(subset=['decade']).groupby('decade').size().reset_index(name='count')
pathe_d['decade'] = pathe_d['decade'].astype(int)
pathe_d['source'] = 'British Pathé (archive)'

imdb_d   = df_imdb.dropna(subset=['decade']).groupby('decade').size().reset_index(name='count')
imdb_d['decade'] = imdb_d['decade'].astype(int)
imdb_d['source'] = 'IMDb (fiction cinema)'

# FCTP — map periods to approximate decades
PERIOD_MAP = {
    'Middle Ages': 1400, '17th Century': 1650, '18th Century': 1750,
    '19th Century': 1850, '1910s–1920s': 1910, '1930s–1940s': 1930,
    '1950s': 1950, '1960s': 1960, '1970s': 1970, '1980s': 1980,
    '1990s': 1990, '20th Century': 1930, '21st Century': 2000,
}
df_f2 = df_fctp.copy()
df_f2['period_simple'] = df_f2['period'].apply(simplify_period)
df_f2['decade'] = df_f2['period_simple'].map(PERIOD_MAP)
fctp_d = df_f2.dropna(subset=['decade']).groupby('decade').size().reset_index(name='count')
fctp_d['decade'] = fctp_d['decade'].astype(int)
fctp_d['source'] = 'FCTP (architecture)'

# Filter to comparable range
combined = pd.concat([pathe_d, imdb_d, fctp_d])
combined = combined[combined['decade'].between(1880, 2030)]

# Normalise each source to 0-1 for fair comparison
for src in combined['source'].unique():
    mask = combined['source'] == src
    mx   = combined.loc[mask, 'count'].max()
    combined.loc[mask, 'count_norm'] = combined.loc[mask, 'count'] / mx

fig, ax = plt.subplots(figsize=(13, 5))

colours = {'British Pathé (archive)': C_PATHE,
           'IMDb (fiction cinema)':   C_IMDB,
           'FCTP (architecture)':     C_FCTP}

for src, grp in combined.groupby('source'):
    grp = grp.sort_values('decade')
    ax.plot(grp['decade'], grp['count_norm'],
            color=colours[src], linewidth=2.2, label=src, marker='o',
            markersize=5, alpha=0.9)
    ax.fill_between(grp['decade'], grp['count_norm'],
                    alpha=0.08, color=colours[src])

ax.set_xlabel('Decade', labelpad=8)
ax.set_ylabel('Normalised activity (0–1)', labelpad=8)
ax.set_title('Turin Across Three Datasets — Shared Timeline\n'
             '(each dataset normalised to its own peak)',
             pad=14, fontweight='bold')
ax.legend(framealpha=0.9, fontsize=10)
ax.set_xlim(1880, 2030)

fig.tight_layout()
save(fig, OUT_CROSS, 'cross_shared_timeline.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\cross_dataset\cross_shared_timeline.png


### 7.2 Cross-dataset semantic category heatmap

This is the most thesis-ready visualisation. I define a shared vocabulary of 10 semantic categories — themes that run across all four datasets — and count how prominently each appears in each source. The result shows what aspects of Turin each medium emphasises, and which are ignored. Architecture says nothing about labour. The archive says nothing about the body. The dialogues say nothing about sport. These silences are as meaningful as the presences.

In [49]:
SEMANTIC = {
    'Industry':      ['factory','fiat','industrial','machine','plant','manufact','produc','workshop'],
    'Labour':        ['strike','worker','union','lay-off','workforce','toil','labor','labour'],
    'Mobility':      ['car','tram','road','traffic','transport','rally','speed','automobile','vehicle'],
    'Conflict':      ['war','bomb','riot','prison','violence','military','brigades','guerrilla','crime'],
    'Spectacle':     ['carnival','festival','show','exhibition','motor show','ceremony','parade','sport'],
    'Monumentality': ['palace','monument','arch','royal','ceremonial','grand','monumental','tower'],
    'Memory':        ['history','past','ancient','archive','memory','heritage','ruin','era','century'],
    'Body / Senses': ['sound','smell','touch','echo','footstep','damp','texture','breath','skin','warmth'],
    'Nature':        ['river','park','garden','tree','green','light','shadow','season','damp','stone'],
    'Domesticity':   ['bench','cafe','table','chair','window','street','neighbourhood','home','shop'],
}

def score_text(text, keywords):
    t = str(text).lower()
    return sum(t.count(k) for k in keywords)

# Compute scores per dataset
datasets = {
    'British Pathé\n(archive)':      ' '.join(df_pathe['title'].fillna('') + ' ' + df_pathe['short_summary'].fillna('')),
    'IMDb\n(cinema)':                ' '.join(df_imdb['title'].fillna('') + ' ' + df_imdb['genres'].fillna('')),
    'FCTP\n(architecture)':          ' '.join(df_fctp['category'].fillna('') + ' ' + df_fctp['style'].fillna('') + ' ' + df_fctp['appearance and condition'].fillna('')),
    'Dialogues\n(Franco+Nietzsche)': ' '.join(df_dial['full'].fillna('')),
}

matrix = {}
for ds_name, ds_text in datasets.items():
    matrix[ds_name] = {cat: score_text(ds_text, kws) for cat, kws in SEMANTIC.items()}

df_sem = pd.DataFrame(matrix).T

# Normalise each column (category) to 0-1 across datasets
df_sem_norm = df_sem.div(df_sem.max(axis=0), axis=1).fillna(0)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(df_sem_norm, ax=ax,
            cmap='RdGy_r', linewidths=0.5, linecolor='white',
            annot=df_sem, fmt='d', annot_kws={'size': 9},
            cbar_kws={'label': 'Relative presence (normalised)', 'shrink': 0.7},
            vmin=0, vmax=1)

ax.set_xlabel('')
ax.set_ylabel('')
ax.set_title('Turin Across Four Datasets — Semantic Category Presence\n'
             '(colour = relative prominence, numbers = raw keyword count)',
             pad=14, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')

fig.tight_layout()
save(fig, OUT_CROSS, 'cross_semantic_category_heatmap.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\cross_dataset\cross_semantic_category_heatmap.png


### 7.3 Dataset overview — size and composition

A simple summary chart showing the scale of each dataset. Useful as an opening figure in the report to orient the reader before the detailed analysis.

dataset_info = {
    'British Pathé\n(archive clips)':     len(df_pathe),
    'IMDb\n(fiction films)':              len(df_imdb),
    'FCTP\n(industrial buildings)':       len(df_fctp),
    'GPT Dialogues\n(locations)':         len(df_dial),
}

labels = list(dataset_info.keys())
values = list(dataset_info.values())
colors = PAL_MIX

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, values, color=colors, alpha=0.85,
              edgecolor='white', linewidth=0.8, width=0.55)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Number of entries', labelpad=8)
ax.set_title('Dataset Overview — Turin Project', pad=14, fontweight='bold')
ax.set_ylim(0, max(values) * 1.15)
ax.grid(axis='y', alpha=0.25, linestyle='--')
ax.set_axisbelow(True)

fig.tight_layout()
save(fig, OUT_CROSS, 'cross_dataset_overview.png')

## 8. Summary of outputs

A final check confirming all visualisations have been saved correctly.

In [39]:
print('All saved outputs:\n')
for folder in [OUT_PATHE, OUT_IMDB, OUT_FCTP, OUT_DIAL, OUT_CROSS]:
    files = [f for f in os.listdir(folder) if f.endswith('.png')]
    folder_name = os.path.basename(folder)
    print(f'  {folder_name}/')
    for f in files:
        print(f'    - {f}')
print(f'\nTotal: {sum(len([f for f in os.listdir(d) if f.endswith(".png")]) for d in [OUT_PATHE, OUT_IMDB, OUT_FCTP, OUT_DIAL, OUT_CROSS])} charts saved')

All saved outputs:

  british_pathe/
    - pathe_clips_per_decade.png
    - pathe_topic_decade_heatmap.png
  imdb/
    - imdb_films_per_decade.png
    - imdb_genre_decade_heatmap.png
  fctp/
    - fctp_style_period_heatmap.png
  dialogues/
    - advanced_diverging_journey.png
    - architectural_route_score.png
    - dialogue_character_voice_comparison.png
  cross_dataset/
    - advanced_radial_timeline.png
    - advanced_route_map.png
    - advanced_theme_network.png
    - architectural_city_section.png
    - architectural_phase_diagram.png
    - cross_dataset_overview.png
    - cross_semantic_category_heatmap.png
    - cross_shared_timeline.png

Total: 16 charts saved


## Better Visualizations: 

In [40]:
!pip install networkx matplotlib seaborn numpy -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\elena\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


## A. Turin Route Map — Semantic Walk
The 24 dialogue locations plotted on Turin's coordinate grid, connected in route order. Each dot is sized by total dialogue word count — places that generated more language are larger. Colour encodes the dominant semantic register of each location: warm tones for sensory-heavy places, cool tones for more philosophical exchanges. The walk becomes a legible diagram of how the city shifts in character as you move through it.

In [41]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# 24 locations with coordinates extracted from the map
ROUTE = [
    {"name": "Statuto",                          "lat": 45.0767, "lon": 7.6701},
    {"name": "Piazza Statuto",                   "lat": 45.0767, "lon": 7.6701},
    {"name": "Café Paris",                       "lat": 45.0705, "lon": 7.6868},
    {"name": "Via Garibaldi",                    "lat": 45.0731, "lon": 7.6794},
    {"name": "Piazza delle Erbe",                "lat": 45.0682, "lon": 7.6821},
    {"name": "Cattedrale San Giovanni",          "lat": 45.0739, "lon": 7.6832},
    {"name": "Porta Palatina",                   "lat": 45.0750, "lon": 7.6847},
    {"name": "Palazzo Reale",                    "lat": 45.0739, "lon": 7.6870},
    {"name": "Piazza Castello",                  "lat": 45.0737, "lon": 7.6865},
    {"name": "Palazzo Madama",                   "lat": 45.0737, "lon": 7.6869},
    {"name": "Palazzo Carignano",                "lat": 45.0733, "lon": 7.6854},
    {"name": "Piazza Carlo Alberto",             "lat": 45.0739, "lon": 7.6870},
    {"name": "Museo Egizio",                     "lat": 45.0730, "lon": 7.6799},
    {"name": "Piazza San Carlo",                 "lat": 45.0673, "lon": 7.6819},
    {"name": "Piazza CLN",                       "lat": 45.0683, "lon": 7.6840},
    {"name": "Piazza Carlina",                   "lat": 45.0689, "lon": 7.6840},
    {"name": "Giardini Cavour",                  "lat": 45.0646, "lon": 7.6857},
    {"name": "Piazza Maria Teresa",              "lat": 45.0636, "lon": 7.6835},
    {"name": "Piazza Vittorio Veneto",           "lat": 45.0636, "lon": 7.6890},
    {"name": "Gran Madre di Dio",                "lat": 45.0621, "lon": 7.6985},
    {"name": "Monumental Arch",                  "lat": 45.0649, "lon": 7.6933},
    {"name": "Castello del Valentino",           "lat": 45.0569, "lon": 7.6863},
    {"name": "Baby Luna Park",                   "lat": 45.0532, "lon": 7.6847},
    {"name": "Borgo Medievale",                  "lat": 45.0486, "lon": 7.6850},
]

# Sensory vs philosophical score per location
# Using keyword counts from the dialogue texts
SENSORY_KW     = ['sound','smell','touch','echo','footstep','damp','texture',
                   'breath','warm','cold','grain','rough','soft','creak','scent']
PHILOSOPH_KW   = ['time','memory','past','void','power','distance','form',
                   'reason','will','spirit','beauty','truth','becoming','weight']

scores = []
word_counts = []
for loc in ROUTE:
    # Match dialogue by location name
    name_clean = loc['name'].lower().replace(' ', '_').replace(',', '').replace('-', '_')
    match = df_dial[df_dial['location'].str.lower().str.replace(' ', '_')
                    .str.replace(',','').str.replace('-','_')
                    .str.contains(name_clean[:8])]
    if len(match) > 0:
        full = match.iloc[0]['full'].lower()
        s = sum(full.count(k) for k in SENSORY_KW)
        p = sum(full.count(k) for k in PHILOSOPH_KW)
        wc = len(full.split())
        ratio = s / (s + p + 1)   # 1 = pure sensory, 0 = pure philosophical
    else:
        ratio = 0.5
        wc    = 200
    scores.append(ratio)
    word_counts.append(wc)

lats  = [l['lat'] for l in ROUTE]
lons  = [l['lon'] for l in ROUTE]
names = [l['name'] for l in ROUTE]
sizes = np.array(word_counts)
sizes = 60 + (sizes - min(sizes)) / (max(sizes) - min(sizes) + 1) * 280

# Colour: warm (sensory) to cool (philosophical)
cmap  = plt.cm.RdYlBu
norm  = Normalize(vmin=0, vmax=1)
colors= [cmap(norm(s)) for s in scores]

fig, ax = plt.subplots(figsize=(9, 13))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#1a1a2e')

# Draw route line
ax.plot(lons, lats, color='#ffffff', linewidth=0.8, alpha=0.25, zorder=1)

# Draw dots
sc = ax.scatter(lons, lats, s=sizes, c=scores, cmap='RdYlBu',
                vmin=0, vmax=1, zorder=3, edgecolors='white',
                linewidth=0.5, alpha=0.92)

# Route order numbers
for i, (lon, lat, name) in enumerate(zip(lons, lats, names)):
    ax.text(lon, lat, str(i+1), ha='center', va='center',
            fontsize=6.5, color='white', fontweight='bold', zorder=4)
    # Label offset
    offset_x = 0.0015 if i % 2 == 0 else -0.0015
    ax.text(lon + offset_x, lat + 0.0008, name,
            fontsize=6, color='#cccccc', ha='center', zorder=4,
            path_effects=[pe.withStroke(linewidth=1.5, foreground='#1a1a2e')])

# Colourbar
cbar = fig.colorbar(sc, ax=ax, fraction=0.025, pad=0.02, shrink=0.4)
cbar.ax.set_ylabel('← Philosophical         Sensory →',
                    color='white', fontsize=8, labelpad=8)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white', fontsize=7)
cbar.outline.set_edgecolor('white')

ax.set_xlabel('Longitude', color='#888', fontsize=9)
ax.set_ylabel('Latitude',  color='#888', fontsize=9)
ax.tick_params(colors='#666')
for spine in ax.spines.values():
    spine.set_edgecolor('#333')

ax.set_title('Turin — A Walk Through Two Ways of Seeing\n'
             'Dot size = dialogue length  |  Colour = sensory ↔ philosophical',
             color='white', fontsize=11, fontweight='bold', pad=14)

fig.tight_layout()
save(fig, OUT_CROSS, 'advanced_route_map.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\cross_dataset\advanced_route_map.png


### B. Cross-Dataset Theme Network

Each dataset is a node. Each semantic theme is a node. Edges connect datasets to the themes they most strongly express. The thickness of each edge encodes the strength of that connection. What emerges is a map of what each medium emphasises and what it ignores — which themes belong to the archive, which to cinema, which to architecture, which to language.

In [42]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

SEMANTIC = {
    'Industry':      ['factory','fiat','industrial','machine','plant','manufact','produc'],
    'Labour':        ['strike','worker','union','lay-off','workforce','toil'],
    'Mobility':      ['car','tram','road','traffic','transport','rally','automobile'],
    'Conflict':      ['war','bomb','riot','prison','violence','military','crime'],
    'Spectacle':     ['carnival','festival','show','exhibition','ceremony','parade','sport'],
    'Monumentality': ['palace','monument','arch','royal','ceremonial','grand','monumental'],
    'Memory':        ['history','past','ancient','archive','memory','heritage','ruin'],
    'Body / Senses': ['sound','smell','touch','echo','footstep','damp','texture','breath'],
    'Nature':        ['river','park','garden','tree','green','shadow','stone','light'],
    'Domesticity':   ['bench','cafe','table','chair','window','neighbourhood','home'],
}

DATASETS = {
    'British Pathé':  ' '.join(df_pathe['title'].fillna('') + ' ' + df_pathe['short_summary'].fillna('')),
    'IMDb Cinema':    ' '.join(df_imdb['title'].fillna('') + ' ' + df_imdb['genres'].fillna('')),
    'Architecture':   ' '.join(df_fctp['category'].fillna('') + ' ' + df_fctp['style'].fillna('') + ' ' + df_fctp['appearance and condition'].fillna('')),
    'Dialogues':      ' '.join(df_dial['full'].fillna('')),
}

DS_COLORS    = {'British Pathé': C_PATHE, 'IMDb Cinema': C_IMDB,
                'Architecture': C_FCTP, 'Dialogues': C_DIAL}
THEME_COLOR  = '#e8e0d4'
THRESHOLD    = 3   # minimum score to draw an edge

G = nx.Graph()

# Add dataset nodes
for ds in DATASETS:
    G.add_node(ds, node_type='dataset', color=DS_COLORS[ds])

# Add theme nodes and edges
for theme, kws in SEMANTIC.items():
    G.add_node(theme, node_type='theme', color=THEME_COLOR)
    for ds, text in DATASETS.items():
        score = sum(text.lower().count(k) for k in kws)
        if score >= THRESHOLD:
            G.add_edge(ds, theme, weight=score)

# Layout — datasets in a square, themes around them
ds_nodes    = [n for n, d in G.nodes(data=True) if d['node_type'] == 'dataset']
theme_nodes = [n for n, d in G.nodes(data=True) if d['node_type'] == 'theme']

pos = {}
# Dataset positions — corners
ds_pos = [(-1.2, 1.2), (1.2, 1.2), (-1.2, -1.2), (1.2, -1.2)]
for ds, p in zip(ds_nodes, ds_pos):
    pos[ds] = p

# Theme positions — circle
n_themes = len(theme_nodes)
for i, t in enumerate(theme_nodes):
    angle = 2 * np.pi * i / n_themes
    pos[t] = (0.65 * np.cos(angle), 0.65 * np.sin(angle))

fig, ax = plt.subplots(figsize=(12, 12))
fig.patch.set_facecolor('#fafaf7')
ax.set_facecolor('#fafaf7')

# Draw edges — thickness and alpha by weight
max_w = max(d['weight'] for _, _, d in G.edges(data=True))
for u, v, data in G.edges(data=True):
    w = data['weight']
    ds = u if G.nodes[u]['node_type'] == 'dataset' else v
    color = DS_COLORS[ds]
    alpha = 0.15 + 0.65 * (w / max_w)
    lw    = 0.5 + 4.0 * (w / max_w)
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    ax.plot([x0, x1], [y0, y1], color=color, linewidth=lw, alpha=alpha, zorder=1)

# Draw theme nodes
for t in theme_nodes:
    x, y = pos[t]
    ax.scatter(x, y, s=220, color='#e8e0d4', edgecolors='#999',
               linewidth=1, zorder=3)
    ax.text(x, y, t, ha='center', va='center', fontsize=8.5,
            color='#333', fontweight='500', zorder=4)

# Draw dataset nodes
for ds in ds_nodes:
    x, y = pos[ds]
    degree = G.degree(ds)
    ax.scatter(x, y, s=1400, color=DS_COLORS[ds], edgecolors='white',
               linewidth=2, zorder=5, alpha=0.9)
    ax.text(x, y, ds, ha='center', va='center', fontsize=9.5,
            color='white', fontweight='bold', zorder=6,
            wrap=True)

# Legend — edge weight
for w_ex, label in [(5, 'weak'), (50, 'medium'), (150, 'strong')]:
    lw = 0.5 + 4.0 * (w_ex / max_w)
    ax.plot([], [], color='#999', linewidth=lw, label=f'{label} connection')
ax.legend(loc='lower right', framealpha=0.9, fontsize=9)

ax.set_xlim(-1.9, 1.9)
ax.set_ylim(-1.7, 1.7)
ax.axis('off')
ax.set_title('Turin — Cross-Dataset Theme Network\n'
             'Edge thickness = connection strength between dataset and theme',
             fontsize=12, fontweight='bold', pad=16, color='#222')

fig.tight_layout()
save(fig, OUT_CROSS, 'advanced_theme_network.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\cross_dataset\advanced_theme_network.png


### C. Radial Timeline

The same timeline data wrapped into a circle. Each decade is a segment of the ring, three concentric bands represent the three datasets. Colour intensity encodes activity level. The radial format makes the comparison between datasets immediate and spatial — you can read the city's history as a clock face.

In [43]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge
from matplotlib.colors import Normalize
import matplotlib.cm as cm

# Decade range
DECADES = list(range(1900, 2030, 10))
N = len(DECADES)

# Count per decade per dataset
def decade_counts(df, decade_col, decades):
    counts = df.dropna(subset=[decade_col]).groupby(decade_col).size()
    return np.array([counts.get(d, 0) for d in decades], dtype=float)

pathe_c  = decade_counts(df_pathe, 'decade', DECADES)
imdb_c   = decade_counts(df_imdb,  'decade', DECADES)

# FCTP — map periods to decades
PERIOD_MAP2 = {
    'Middle Ages': 1400, '17th Century': 1650, '18th Century': 1750,
    '19th Century': 1850, '1910s–1920s': 1910, '1930s–1940s': 1930,
    '1950s': 1950, '1960s': 1960, '1970s': 1970, '1980s': 1980,
    '1990s': 1990, '20th Century': 1930, '21st Century': 2000,
}
df_f3 = df_fctp.copy()
df_f3['period_simple'] = df_f3['period'].apply(simplify_period)
df_f3['decade2'] = df_f3['period_simple'].map(PERIOD_MAP2)
fctp_c = decade_counts(df_f3, 'decade2', DECADES)

# Normalise each to 0-1
def norm01(arr):
    mx = arr.max()
    return arr / mx if mx > 0 else arr

p_n = norm01(pathe_c)
i_n = norm01(imdb_c)
f_n = norm01(fctp_c)

fig, ax = plt.subplots(figsize=(12, 12), subplot_kw={'projection': 'polar'})
fig.patch.set_facecolor('#0f0f1a')
ax.set_facecolor('#0f0f1a')

theta      = np.linspace(0, 2 * np.pi, N, endpoint=False)
width      = (2 * np.pi / N) * 0.82

# Three rings — inner to outer: FCTP, Pathé, IMDb
RINGS = [
    {'data': f_n, 'r_inner': 1.5, 'r_outer': 2.4, 'cmap': 'Greens',  'label': 'Architecture (FCTP)'},
    {'data': p_n, 'r_inner': 2.6, 'r_outer': 3.5, 'cmap': 'Oranges', 'label': 'Archive (Pathé)'},
    {'data': i_n, 'r_inner': 3.7, 'r_outer': 4.6, 'cmap': 'Blues',   'label': 'Cinema (IMDb)'},
]

for ring in RINGS:
    cmap_r = cm.get_cmap(ring['cmap'])
    for j, (t, val) in enumerate(zip(theta, ring['data'])):
        if val > 0.01:
            color = cmap_r(0.3 + 0.7 * val)
            bar = ax.bar(t, ring['r_outer'] - ring['r_inner'],
                         width=width, bottom=ring['r_inner'],
                         color=color, edgecolor='#0f0f1a',
                         linewidth=0.3, alpha=0.92)

# Decade labels
for j, (t, dec) in enumerate(zip(theta, DECADES)):
    angle_deg = np.degrees(t)
    ha = 'left' if angle_deg < 180 else 'right'
    ax.text(t, 5.0, str(dec), ha='center', va='center',
            fontsize=8, color='#aaaaaa', rotation=angle_deg - 90)

# Ring labels
for ring in RINGS:
    r_mid = (ring['r_inner'] + ring['r_outer']) / 2
    ax.text(np.pi * 1.62, r_mid, ring['label'],
            ha='center', va='center', fontsize=8,
            color='white', fontweight='bold')

ax.set_ylim(0, 5.5)
ax.axis('off')
ax.set_title('Turin — Radial Timeline\nThree datasets across the 20th century',
             color='white', fontsize=13, fontweight='bold', pad=20, y=1.04)

fig.tight_layout()
save(fig, OUT_CROSS, 'advanced_radial_timeline.png')

C:\Users\elena\AppData\Local\Temp\ipykernel_9196\655458684.py:55: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_r = cm.get_cmap(ring['cmap'])


Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\cross_dataset\advanced_radial_timeline.png


### D. Franco × Nietzsche — Diverging Journey

The 24 locations plotted as a sequence, with Franco's sensory density above the centre line and Nietzsche's philosophical density below it. Where the lines diverge widely, the two characters experience the place most differently. Where they converge, the place affects both in similar ways. It reads the walk as a score — a notation of how two kinds of perception move through the city.

In [44]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

SENSORY_KW   = ['sound','smell','touch','echo','footstep','damp','texture',
                 'breath','warm','cold','grain','rough','soft','creak','scent',
                 'hear','feel','finger','stone','air','dark','light']
PHILOSOPH_KW = ['time','memory','past','void','power','distance','form',
                 'reason','will','spirit','beauty','truth','becoming','weight',
                 'thought','city','human','life','death','soul','heart','world']

locations_ordered = [
    'Statuto', 'Piazza_Statuto', 'Café_Paris', 'Via_Giuseppe_Garibaldi',
    'Piazza_delle_Erbe', 'Cattedrale_di_San_Giovanni_Batti',
    'Porta_Palatina', 'Palazzo_Reale_di_Torino', 'Piazza_Castello',
    'Palazzo_Madama', 'Palazzo_Carignano', 'Piazza_Carlo_Alberto',
    'Museo_Egizio_(outside)', 'Piazza_San_Carlo', 'Piazza_CLN',
    'Piazza_Carlo_Emanuele_II_(Piazza', 'Giardini_Cavour',
    'Piazza_Maria_Teresa', 'Piazza_Vittorio_Veneto',
    'Chiesa_della_Gran_Madre_di_Dio_(', 'Monumental_Arch_to_the_Artillery',
    'Castello_del_Valentino', 'Il_posto_giusto_del_Baby_Luna_Pa',
    'Borgo_Medievale_(Parco_del_Valen'
]

short_names = [
    'Statuto', 'P. Statuto', 'Café Paris', 'Via Garibaldi',
    'P. Erbe', 'Cattedrale', 'P. Palatina', 'P. Reale',
    'P. Castello', 'P. Madama', 'P. Carignano', 'P. Carlo Alberto',
    'Museo Egizio', 'P. San Carlo', 'P. CLN', 'P. Carlina',
    'G. Cavour', 'P. M. Teresa', 'P. Vittorio', 'Gran Madre',
    'Mon. Arch', 'C. Valentino', 'Luna Park', 'Borgo Med.'
]

franco_scores    = []
nietzsche_scores = []

for loc_id in locations_ordered:
    match = df_dial[df_dial['location'].str.lower().str.replace(' ','_')
                    .str.startswith(loc_id.lower()[:10])]
    if len(match) > 0:
        f_text = match.iloc[0]['franco'].lower()
        n_text = match.iloc[0]['nietzsche'].lower()
        f_wc   = max(len(f_text.split()), 1)
        n_wc   = max(len(n_text.split()), 1)
        f_s    = sum(f_text.count(k) for k in SENSORY_KW)   / f_wc * 100
        n_p    = sum(n_text.count(k) for k in PHILOSOPH_KW) / n_wc * 100
    else:
        f_s, n_p = 2.0, 2.0
    franco_scores.append(f_s)
    nietzsche_scores.append(n_p)

x   = np.arange(len(locations_ordered))
f_s = np.array(franco_scores)
n_p = np.array(nietzsche_scores)

# Smooth slightly
from scipy.ndimage import uniform_filter1d
try:
    f_smooth = uniform_filter1d(f_s, size=2)
    n_smooth = uniform_filter1d(n_p, size=2)
except ImportError:
    f_smooth, n_smooth = f_s, n_p

fig, ax = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor('#fafaf7')
ax.set_facecolor('#fafaf7')

# Fill areas
ax.fill_between(x,  f_smooth, 0, alpha=0.25, color=C_DIAL, label='Franco — sensory density')
ax.fill_between(x, -n_smooth, 0, alpha=0.25, color=C_PATHE, label='Nietzsche — philosophical density')

# Lines
ax.plot(x,  f_smooth, color=C_DIAL,  linewidth=2.2, alpha=0.9)
ax.plot(x, -n_smooth, color=C_PATHE, linewidth=2.2, alpha=0.9)

# Centre line
ax.axhline(0, color='#999', linewidth=0.8, linestyle='--', alpha=0.5)

# Location labels
ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=55, ha='right', fontsize=8)

# Y-axis labels
ax.set_ylabel('← Nietzsche (philosophical)     Franco (sensory) →',
              fontsize=9, labelpad=8)

# Highlight divergence
divergence = np.abs(f_smooth + n_smooth)
peak_idx   = np.argmax(divergence)
ax.annotate(f'Greatest divergence\n{short_names[peak_idx]}',
            xy=(peak_idx, f_smooth[peak_idx]),
            xytext=(peak_idx - 2, f_smooth[peak_idx] + 1.5),
            fontsize=8, color='#555',
            arrowprops=dict(arrowstyle='->', color='#999', lw=0.8))

ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
ax.set_title('Franco × Nietzsche — A Diverging Journey Through Turin\n'
             'How two kinds of perception move through the same 24 locations',
             fontsize=12, fontweight='bold', pad=12)

fig.tight_layout()
save(fig, OUT_DIAL, 'advanced_diverging_journey.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\dialogues\advanced_diverging_journey.png


### 1. Stratified City Section

Turin read as a geological section. Each dataset is a stratum — built form at the base, then archive footage, then cinema, then synthetic dialogue at the surface. Time runs left to right. The thickness of each band at each decade reflects activity level. The city is not measured here; it is excavated.

In [50]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from scipy.ndimage import uniform_filter1d

DECADES = list(range(1880, 2030, 10))
N = len(DECADES)
x = np.arange(N)

def get_decade_counts(df, decade_col, decades):
    counts = df.dropna(subset=[decade_col]).groupby(decade_col).size()
    return np.array([counts.get(float(d), counts.get(int(d), 0)) for d in decades], dtype=float)

pathe_c = get_decade_counts(df_pathe, 'decade', DECADES)
imdb_c  = get_decade_counts(df_imdb,  'decade', DECADES)

df_f3 = df_fctp.copy()
df_f3['period_simple'] = df_f3['period'].apply(simplify_period)
PERIOD_MAP3 = {
    'Middle Ages': 1400, '17th Century': 1650, '18th Century': 1750,
    '19th Century': 1850, '1910s–1920s': 1910, '1930s–1940s': 1930,
    '1950s': 1950, '1960s': 1960, '1970s': 1970, '1980s': 1980,
    '1990s': 1990, '20th Century': 1930, '21st Century': 2000,
}
df_f3['decade_n'] = df_f3['period_simple'].map(PERIOD_MAP3)
fctp_c = get_decade_counts(df_f3, 'decade_n', DECADES)

# Dialogue — flat contemporary layer
dial_c = np.zeros(N)
dial_c[-3:] = len(df_dial) / 3

# Smooth
def smooth(arr, w=2):
    return uniform_filter1d(arr.astype(float), size=w)

layers = [
    {'data': smooth(fctp_c),  'label': 'Architecture\n(built form)',
     'color': '#6BAB7E', 'alpha': 0.82, 'hatch': '....'},
    {'data': smooth(pathe_c), 'label': 'Archive\n(British Pathé)',
     'color': '#C17D5A', 'alpha': 0.75, 'hatch': '////'},
    {'data': smooth(imdb_c),  'label': 'Cinema\n(IMDb)',
     'color': '#5A7DC1', 'alpha': 0.72, 'hatch': '----'},
    {'data': smooth(dial_c),  'label': 'Dialogue\n(synthetic)',
     'color': '#8B2E16', 'alpha': 0.85, 'hatch': ''},
]

# Normalise each layer independently
for layer in layers:
    mx = layer['data'].max()
    layer['norm'] = layer['data'] / mx if mx > 0 else layer['data']

# Stack thickness — each layer gets a base height + variable part
BASE    = 0.8
VAR_MAX = 1.8
SPACING = 0.15

fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor('#fdfcf8')
ax.set_facecolor('#fdfcf8')

y_bottom = np.zeros(N)
for layer in layers:
    heights = BASE + layer['norm'] * VAR_MAX
    # Irregular top edge — add slight noise
    noise   = np.random.default_rng(42).normal(0, 0.04, N)
    heights = heights + noise
    ax.fill_between(x, y_bottom, y_bottom + heights,
                    color=layer['color'], alpha=layer['alpha'],
                    linewidth=0)
    # Thin top contour line
    ax.plot(x, y_bottom + heights, color=layer['color'],
            linewidth=0.8, alpha=0.6)
    # Layer label on right
    mid_y = (y_bottom + y_bottom + heights).mean() / 2 + y_bottom.mean()
    ax.text(N - 0.3, y_bottom.mean() + BASE/2 + layer['norm'].mean() * VAR_MAX/2,
            layer['label'], va='center', ha='left',
            fontsize=8, color=layer['color'],
            fontweight='bold')
    y_bottom = y_bottom + heights + SPACING

# Decade labels
label_idx = [i for i, d in enumerate(DECADES) if d % 20 == 0]
ax.set_xticks([i for i in label_idx])
ax.set_xticklabels([str(DECADES[i]) for i in label_idx],
                    fontsize=9, color='#555')

# Annotation for key events
events = [
    (DECADES.index(1920) if 1920 in DECADES else 4, 'FIAT expansion'),
    (DECADES.index(1940) if 1940 in DECADES else 6, 'WWII'),
    (DECADES.index(1970) if 1970 in DECADES else 9, 'Labour unrest'),
]
for idx, label in events:
    if 0 <= idx < N:
        ax.axvline(idx, color='#aaa', linewidth=0.5,
                   linestyle=':', alpha=0.6, zorder=0)
        ax.text(idx + 0.1, 0.2, label, fontsize=7,
                color='#888', rotation=90, va='bottom')

ax.set_xlim(-0.5, N - 0.2)
ax.set_ylim(0)
ax.set_ylabel('Stratum thickness\n(relative activity)', fontsize=9, color='#555')
ax.set_xlabel('Decade', fontsize=9, color='#555')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#ccc')
ax.spines['bottom'].set_color('#ccc')
ax.tick_params(colors='#888')
ax.yaxis.set_visible(False)

ax.set_title('Turin — Stratified City Section\n'
             'The city read as geological strata: built form, archive, cinema, synthetic perception',
             fontsize=12, fontweight='bold', pad=14, color='#222')

fig.tight_layout()
save(fig, OUT_CROSS, 'architectural_city_section.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\cross_dataset\architectural_city_section.png


### 2. Psychogeographic Route Score

The 24-location walk notated as a score. The route runs left to right. Franco's sensory voice occupies the upper register, Nietzsche's philosophical voice the lower. Line thickness encodes intensity, not just height. Vertical marks punctuate key moments — locations where the two characters diverge most sharply. It reads like a transit map crossed with a musical notation sheet.

In [46]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from scipy.ndimage import uniform_filter1d

SENSORY_KW   = ['sound','smell','touch','echo','footstep','damp','texture',
                 'breath','warm','cold','grain','rough','soft','creak','scent',
                 'hear','feel','finger','stone','air','dark','clink','whisper']
PHILOSOPH_KW = ['time','memory','past','void','power','distance','form',
                 'reason','will','spirit','beauty','truth','becoming','weight',
                 'thought','human','life','death','soul','history','proportion']

LOC_NAMES = [
    'Statuto', 'P.Statuto', 'Café Paris', 'Via Garibaldi',
    'P.Erbe', 'Cattedrale', 'P.Palatina', 'P.Reale',
    'P.Castello', 'P.Madama', 'P.Carignano', 'P.C.Alberto',
    'Museo Egizio', 'P.San Carlo', 'P.CLN', 'P.Carlina',
    'G.Cavour', 'P.M.Teresa', 'P.Vittorio', 'Gran Madre',
    'Mon.Arch', 'C.Valentino', 'Luna Park', 'Borgo Med.'
]

CATEGORIES = {
    'sound':      ['sound','echo','noise','ring','creak','hum','buzz','clink'],
    'material':   ['stone','brick','wood','metal','glass','iron','marble'],
    'industry':   ['factory','fiat','machine','work','labor','forge','wheel'],
    'memory':     ['past','history','ancient','ruin','archive','memory','trace'],
    'body':       ['touch','breath','hand','foot','skin','damp','warm','cold'],
    'void':       ['silence','empty','void','hollow','dark','shadow','still'],
}

f_scores, n_scores, categories_per_loc = [], [], []

for name in LOC_NAMES:
    key = name.lower().replace('.','').replace(' ','_')
    match = df_dial[df_dial['location'].str.lower()
                    .str.replace(' ','_').str.replace(',','')
                    .str.replace('-','_').str.contains(key[:6])]
    if len(match) > 0:
        f_t = match.iloc[0]['franco'].lower()
        n_t = match.iloc[0]['nietzsche'].lower()
        fw  = max(len(f_t.split()), 1)
        nw  = max(len(n_t.split()), 1)
        fs  = sum(f_t.count(k) for k in SENSORY_KW)   / fw * 100
        ns  = sum(n_t.count(k) for k in PHILOSOPH_KW) / nw * 100
        full_t = match.iloc[0]['full'].lower()
        dom = max(CATEGORIES, key=lambda c: sum(full_t.count(k) for k in CATEGORIES[c]))
    else:
        fs, ns, dom = 2.0, 2.0, 'memory'
    f_scores.append(fs)
    n_scores.append(ns)
    categories_per_loc.append(dom)

f_arr = uniform_filter1d(np.array(f_scores), size=2)
n_arr = uniform_filter1d(np.array(n_scores), size=2)
x     = np.arange(len(LOC_NAMES))

CAT_COLORS = {
    'sound':    '#D4956F',
    'material': '#8FA8C1',
    'industry': '#C17D5A',
    'memory':   '#9B7DC1',
    'body':     '#E8C4A0',
    'void':     '#6B8B9E',
}

fig, ax = plt.subplots(figsize=(20, 7))
fig.patch.set_facecolor('#fdfcf8')
ax.set_facecolor('#fdfcf8')

# Centre axis
ax.axhline(0, color='#cccccc', linewidth=0.6, zorder=1)

# Draw score as segments with varying thickness and colour
for i in range(len(x) - 1):
    cat   = categories_per_loc[i]
    color = CAT_COLORS.get(cat, '#aaa')
    # Franco upper
    lw_f = 1.0 + f_arr[i] * 1.8
    ax.plot([x[i], x[i+1]], [f_arr[i], f_arr[i+1]],
            color=color, linewidth=lw_f, alpha=0.85, solid_capstyle='round')
    # Nietzsche lower
    lw_n = 1.0 + n_arr[i] * 1.8
    ax.plot([x[i], x[i+1]], [-n_arr[i], -n_arr[i+1]],
            color=color, linewidth=lw_n, alpha=0.65,
            linestyle=(0, (4, 2)), solid_capstyle='round')

# Fill between
ax.fill_between(x,  f_arr, 0, alpha=0.06, color='#9B7DC1')
ax.fill_between(x, -n_arr, 0, alpha=0.06, color='#C17D5A')

# Category dots on centre line
for i, (xi, cat) in enumerate(zip(x, categories_per_loc)):
    ax.scatter(xi, 0, s=28, color=CAT_COLORS.get(cat, '#aaa'),
               zorder=4, edgecolors='white', linewidth=0.4)

# Vertical bars at high-divergence moments
divergence = f_arr + n_arr
top3 = np.argsort(divergence)[-3:]
for idx in top3:
    ax.axvline(x[idx], color='#aaa', linewidth=0.5, linestyle=':', alpha=0.7)

# Location labels
for i, (xi, name) in enumerate(zip(x, LOC_NAMES)):
    ax.text(xi, -n_arr[i] - 0.5, name,
            ha='center', va='top', fontsize=6.5,
            color='#666', rotation=45)

# Register labels
ax.text(-0.7,  f_arr.mean(), 'FRANCO\nsensory',
        ha='right', va='center', fontsize=8,
        color='#9B7DC1', fontweight='bold')
ax.text(-0.7, -n_arr.mean(), 'NIETZSCHE\nphilosophical',
        ha='right', va='center', fontsize=8,
        color='#C17D5A', fontweight='bold')

# Category legend
handles = [mpatches.Patch(color=v, label=k, alpha=0.85)
           for k, v in CAT_COLORS.items()]
ax.legend(handles=handles, loc='upper right',
          title='Dominant theme', title_fontsize=8,
          fontsize=7.5, framealpha=0.9, ncol=3)

ax.set_xlim(-1.2, len(LOC_NAMES) - 0.5)
ax.set_xticks([])
ax.yaxis.set_visible(False)
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title('Psychogeographic Route Score — Turin Walk\n'
             'Franco (upper, solid) and Nietzsche (lower, dashed) '
             '| line thickness = intensity | colour = dominant theme',
             fontsize=11, fontweight='bold', pad=14, color='#222')

fig.tight_layout()
save(fig, OUT_DIAL, 'architectural_route_score.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\dialogues\architectural_route_score.png


### 3. Ruin–Modernity Phase Diagram

A shared coordinate space where all four datasets can coexist. The x-axis runs from industrial to ceremonial; the y-axis from ruin to modernity. Buildings, Pathé clips, films, and dialogue locations are all plotted as different symbols in the same field. The result is not a comparison of datasets but a shared conceptual atlas — a map of where different representations of Turin cluster in the same abstract space.

In [47]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)

fig, ax = plt.subplots(figsize=(13, 13))
fig.patch.set_facecolor('#fdfcf8')
ax.set_facecolor('#fdfcf8')

# Quadrant shading
ax.fill_between([-1, 0], [0, 0], [1,  1], alpha=0.05, color='#6BAB7E')
ax.fill_between([ 0, 1], [0, 0], [1,  1], alpha=0.05, color='#5A7DC1')
ax.fill_between([-1, 0], [-1,-1], [0, 0], alpha=0.05, color='#C17D5A')
ax.fill_between([ 0, 1], [-1,-1], [0, 0], alpha=0.05, color='#9B7DC1')
ax.axhline(0, color='#ccc', linewidth=0.7, zorder=1)
ax.axvline(0, color='#ccc', linewidth=0.7, zorder=1)

# --- FCTP buildings: position by period and condition ---
PERIOD_X = {
    'Middle Ages': -0.7, '17th Century': -0.55, '18th Century': -0.4,
    '19th Century': -0.25, '1910s–1920s': -0.1, '1930s–1940s': 0.05,
    '1950s': 0.15, '1960s': 0.2, '1970s': 0.25, '1980s': 0.3,
    '1990s': 0.35, '20th Century': 0.1, '21st Century': 0.45, 'Unknown': 0.0
}
CONDITION_Y = {
    'Renovated': 0.7, 'Modern': 0.65, 'Modern, Renovated': 0.75,
    'Traditional': 0.2, 'Original (not renovated)': -0.1,
    'Alternative': 0.3, 'Dilapidated': -0.6, 'Abandoned': -0.8,
    'Run down': -0.7, 'Empty': -0.5, 'Working-class': -0.2,
    'Unknown': 0.0, '-': 0.0,
}

for _, row in df_fctp.iterrows():
    period  = simplify_period(str(row.get('period', '')))
    cond    = str(row.get('appearance and condition', '-')).split(',')[0].strip()
    base_x  = PERIOD_X.get(period, 0.0)
    base_y  = CONDITION_Y.get(cond, 0.0)
    x = np.clip(base_x + np.random.uniform(-0.12, 0.12), -0.95, 0.95)
    y = np.clip(base_y + np.random.uniform(-0.1,  0.1),  -0.95, 0.95)
    ax.scatter(x, y, marker='s', s=55, color=C_FCTP,
               alpha=0.78, edgecolors='white', linewidth=0.4, zorder=3)
    ax.text(x + 0.015, y, str(row['title'])[:14],
            fontsize=5.5, color='#4a7a5a', alpha=0.8, va='center', zorder=4)

# --- Pathé clips: position by topic and year ---
TOPIC_X = {
    'Labour / Strikes': -0.65, 'Industry / Cars': -0.5,
    'War / Military': -0.35, 'Crime / Unrest': -0.2,
    'Sport': 0.1, 'Culture / Events': 0.4,
    'Royalty / Politics': 0.6, 'Other': 0.0,
}
for _, row in df_pathe.iterrows():
    topic  = row.get('topic', 'Other')
    year   = row.get('year', 1950)
    base_x = TOPIC_X.get(topic, 0.0)
    # y: older = more ruin-like, newer = more modern
    base_y = np.clip((float(year) - 1940) / 40, -0.85, 0.85) if year else 0.0
    x = np.clip(base_x + np.random.uniform(-0.15, 0.15), -0.95, 0.95)
    y = np.clip(base_y + np.random.uniform(-0.1,  0.1),  -0.95, 0.95)
    ax.scatter(x, y, marker='o', s=20, color=C_PATHE,
               alpha=0.65, edgecolors='none', zorder=2)

# --- IMDb films: position by decade and genre ---
GENRE_X = {
    'Documentary': -0.3, 'Short': 0.0, 'Drama': 0.1,
    'Crime': -0.15, 'Thriller': -0.2, 'Comedy': 0.3,
    'Biography': 0.2, 'History': 0.0, 'Romance': 0.35,
    'Music': 0.25, 'Horror': -0.4, 'Action': -0.35,
}
for _, row in df_imdb.iterrows():
    genre  = str(row.get('genres', '')).split(',')[0].strip()
    year   = row.get('year', 1970)
    base_x = GENRE_X.get(genre, 0.0)
    base_y = np.clip((float(year) - 1960) / 40, -0.85, 0.85) if year else 0.0
    x = np.clip(base_x + np.random.uniform(-0.18, 0.18), -0.95, 0.95)
    y = np.clip(base_y + np.random.uniform(-0.12, 0.12), -0.95, 0.95)
    ax.scatter(x, y, marker='^', s=20, color=C_IMDB,
               alpha=0.60, edgecolors='none', zorder=2)

# --- Dialogues: position by semantic character ---
DIAL_POSITIONS = {
    'Statuto': (-0.65, -0.3), 'Piazza_Statuto': (-0.55, -0.2),
    'Café_Paris': (0.3, 0.4), 'Via_Giuseppe_Garibaldi': (-0.1, 0.1),
    'Piazza_delle_Erbe': (0.15, -0.1), 'Cattedrale': (-0.1, -0.5),
    'Porta_Palatina': (-0.4, -0.7), 'Palazzo_Reale': (0.7, 0.3),
    'Piazza_Castello': (0.6, 0.2), 'Palazzo_Madama': (0.5, 0.1),
    'Palazzo_Carignano': (0.4, 0.0), 'Piazza_Carlo_Alberto': (0.3, -0.1),
    'Museo_Egizio': (0.2, -0.3), 'Piazza_San_Carlo': (0.55, 0.15),
    'Piazza_CLN': (0.1, -0.2), 'Piazza_Carlo_Emanuele': (0.2, 0.1),
    'Giardini_Cavour': (-0.2, 0.3), 'Piazza_Maria_Teresa': (0.0, 0.2),
    'Piazza_Vittorio': (0.35, -0.15), 'Chiesa_della_Gran': (0.15, -0.55),
    'Monumental_Arch': (-0.3, -0.4), 'Castello_del_Valentino': (-0.15, 0.2),
    'Il_posto_giusto': (0.1, 0.5), 'Borgo_Medievale': (-0.5, -0.6),
}

SHORT_NAMES = {
    'Statuto': 'Statuto', 'Piazza_Statuto': 'P. Statuto',
    'Café_Paris': 'Café Paris', 'Via_Giuseppe_Garibaldi': 'Via Garibaldi',
    'Piazza_delle_Erbe': 'P. Erbe', 'Cattedrale': 'Cattedrale',
    'Porta_Palatina': 'P. Palatina', 'Palazzo_Reale': 'P. Reale',
    'Piazza_Castello': 'P. Castello', 'Palazzo_Madama': 'P. Madama',
    'Palazzo_Carignano': 'P. Carignano', 'Piazza_Carlo_Alberto': 'P. C. Alberto',
    'Museo_Egizio': 'Museo Egizio', 'Piazza_San_Carlo': 'P. San Carlo',
    'Piazza_CLN': 'P. CLN', 'Piazza_Carlo_Emanuele': 'P. Carlina',
    'Giardini_Cavour': 'G. Cavour', 'Piazza_Maria_Teresa': 'P. M. Teresa',
    'Piazza_Vittorio': 'P. Vittorio', 'Chiesa_della_Gran': 'Gran Madre',
    'Monumental_Arch': 'Mon. Arch', 'Castello_del_Valentino': 'C. Valentino',
    'Il_posto_giusto': 'Luna Park', 'Borgo_Medievale': 'Borgo Med.',
}

for _, row in df_dial.iterrows():
    loc = str(row['location'])
    key = next((k for k in DIAL_POSITIONS if loc.startswith(k[:10])), None)
    if key:
        bx, by = DIAL_POSITIONS[key]
    else:
        bx, by = np.random.uniform(-0.3, 0.3), np.random.uniform(-0.3, 0.3)
    x = np.clip(bx + np.random.uniform(-0.04, 0.04), -0.95, 0.95)
    y = np.clip(by + np.random.uniform(-0.04, 0.04), -0.95, 0.95)
    label = SHORT_NAMES.get(key, loc[:12]) if key else loc[:12]
    ax.scatter(x, y, marker='D', s=60, color=C_DIAL,
               alpha=0.82, edgecolors='white', linewidth=0.5, zorder=5)
    ax.text(x + 0.02, y + 0.02, label,
            fontsize=7, color='#6B4C9B', fontweight='500', zorder=6)

# Axis labels and quadrant text
for txt, kw in [('ceremonial →', dict(x=0.98, y=-0.03, ha='right')),
                ('← industrial', dict(x=0.02, y=-0.03, ha='left'))]:
    ax.text(transform=ax.transAxes, s=txt, fontsize=9,
            color='#999', style='italic', **kw)

ax.text(0.01, 0.97, 'modernity ↑', transform=ax.transAxes,
        ha='left', va='top', fontsize=9, color='#999',
        style='italic', rotation=90)
ax.text(0.01, 0.03, '↓ ruin', transform=ax.transAxes,
        ha='left', va='bottom', fontsize=9, color='#999',
        style='italic', rotation=90)

for txt, x, y, col in [
    ('Industrial\nModern', -0.85, 0.80, '#6BAB7E'),
    ('Ceremonial\nModern', 0.50,  0.80, '#5A7DC1'),
    ('Industrial\nRuin',   -0.85, -0.90, '#C17D5A'),
    ('Ceremonial\nRuin',    0.50, -0.90, '#9B7DC1'),
]:
    ax.text(x, y, txt, fontsize=9, color=col,
            alpha=0.45, fontweight='bold')

legend_elements = [
    mpatches.Patch(color=C_FCTP,  label='Architecture (FCTP)', alpha=0.8),
    mpatches.Patch(color=C_PATHE, label='British Pathé archive', alpha=0.8),
    mpatches.Patch(color=C_IMDB,  label='IMDb cinema', alpha=0.8),
    mpatches.Patch(color=C_DIAL,  label='Dialogues (locations)', alpha=0.8),
]
ax.legend(handles=legend_elements, loc='lower right',
          framealpha=0.92, fontsize=9)

ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-1.05, 1.05)
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_color('#e0e0e0')

ax.set_title('Turin — Ruin–Modernity Phase Diagram\n'
             'All four datasets in a shared conceptual field',
             fontsize=12, fontweight='bold', pad=14, color='#222')

fig.tight_layout()
save(fig, OUT_CROSS, 'architectural_phase_diagram.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VISUALIZING\OUTPUTS\cross_dataset\architectural_phase_diagram.png
